# Practical PyTorch · Part 5 — Companion Notebook

This goes with **"Your First Real Model Run, From Photo to Verdict"**. Run the cells top to bottom (Shift+Enter).

We'll download a photo and classify it with a pretrained **ResNet-50** from torchvision. No GPU needed — this runs fine on CPU.

## 0. The easy button (a ten-second taste)

Before the by-hand image run, here's how easy a model *can* be when it's all wrapped up for you — the three-liner from Part 1, run for real. Three lines, a real model, a real answer, and no idea what happened inside. That's the catch we spend the rest of the notebook fixing. (`transformers` isn't in Colab by default, so install it once.)

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import pipeline

classify = pipeline('sentiment-analysis')
print(classify("I can't believe how easy this was."))
# [{'label': 'POSITIVE', 'score': 0.9998}]

## 1. Load the model and its weights

`ResNet50_Weights.DEFAULT` = the best available pretrained weights. The first run downloads them (a few seconds), then caches them. `.eval()` puts the model in evaluation mode — always do this for inference.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights).eval()

print("Model loaded. It knows", len(weights.meta["categories"]), "categories.")

## 2. Get an image

We download a photo from Wikimedia Commons and open it with Pillow. `.convert("RGB")` guarantees three color channels, which is what the model expects.

In [ ]:
import requests
from PIL import Image

url = "https://upload.wikimedia.org/wikipedia/commons/2/26/YellowLabradorLooking_new.jpg"
img = Image.open(requests.get(url, stream=True).raw).convert("RGB")

img  # Colab will display it

## 3. Preprocess (let the weights do it)

`weights.transforms()` returns the exact resize-crop-normalize pipeline this model was trained with — so it can't drift out of sync. `.unsqueeze(0)` adds the batch dimension the model expects.

In [ ]:
preprocess = weights.transforms()

batch = preprocess(img).unsqueeze(0)
print("batch shape:", batch.shape)  # [1, 3, 224, 224] — batch, channels, height, width

## 4. Run it (eval + no_grad)

`torch.no_grad()` turns off training-only bookkeeping we don't need — faster and lighter. The output is 1,000 raw scores, one per category.

In [ ]:
import torch

with torch.no_grad():
    out = model(batch)

print("output shape:", out.shape)  # [1, 1000]

## 5. Read the answer

`softmax` turns raw scores into probabilities; `argmax` picks the winner; `weights.meta["categories"]` maps that index to a human-readable label.

In [ ]:
probs = out.softmax(dim=1)[0]
class_id = probs.argmax().item()

label = weights.meta["categories"][class_id]
confidence = probs[class_id].item()

print(f"{label}  ({confidence:.1%})")

## Bonus: the top 5 guesses

The single winner is rarely the whole story. Here are the model's five most confident guesses.

In [ ]:
top_probs, top_ids = probs.topk(5)

for p, i in zip(top_probs, top_ids):
    print(f"{weights.meta['categories'][i]:30s} {p.item():.1%}")

## Your turn

Swap the `url` in cell 2 for any image you like (try a cat, a sports car, a coffee mug) and re-run from there. Same five steps, every time: **load, prepare, infer, read.**